<a href="https://colab.research.google.com/github/alirezakavianifar/gitTutorial/blob/developer/browse_and_concentrate_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import gc
import tensorflow as tf
from tensorflow.keras import mixed_precision
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
import cv2
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, TFBertModel
from tensorflow.keras.layers import (
    Input, Dense, Conv1D, GlobalAvgPool1D, GlobalAvgPool2D,
    MultiHeadAttention, LayerNormalization, Concatenate, Reshape,
    Add, Lambda
)
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras import backend as K

# Clear any existing TensorFlow sessions
tf.keras.backend.clear_session()

# Configure GPU memory more conservatively
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set memory limit to 4GB (adjust based on your needs)
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=4096)]
        )
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
    except RuntimeError as e:
        print(e)

# Enable mixed precision training
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Clear memory
gc.collect()

# Define constants
im_size = (128, 128)  # Reduced from (224, 224)
im_size_final = (im_size[0], im_size[1], 3)  # Add channel dimension
input_size = im_size_final
batch_size = 4  # Reduced from 8
maxlen = 128  # Reduced from 500

class LazyImageLoader:
    def __init__(self, image_dir, image_names, target_size):
        self.image_dir = image_dir
        self.image_names = image_names
        self.target_size = target_size
        self.cache = {}
        self.max_cache_size = 1000  # Limit cache size

    def _load_image(self, img_name):
        img_path = os.path.join(self.image_dir, f"{img_name}.png")
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, self.target_size)
                img = img.astype(np.float16) / 255.0
                return img
        return np.zeros((*self.target_size, 3), dtype=np.float16)

    def _load_batch(self, indices):
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(self._load_image, self.image_names[idx]) for idx in indices]
            return [f.result() for f in futures]

    def __getitem__(self, idx):
        if idx in self.cache:
            return self.cache[idx]

        # Load a batch of images around the requested index
        batch_size = 32
        start_idx = max(0, idx - batch_size // 2)
        end_idx = min(len(self.image_names), idx + batch_size // 2)
        batch_indices = range(start_idx, end_idx)

        # Load the batch
        batch_images = self._load_batch(batch_indices)

        # Cache the results
        for i, img_idx in enumerate(batch_indices):
            self.cache[img_idx] = batch_images[i]

            # Remove oldest cache entries if cache is full
            if len(self.cache) > self.max_cache_size:
                oldest_key = next(iter(self.cache))
                del self.cache[oldest_key]

        return self.cache[idx]

    def __len__(self):
        return len(self.image_names)

class MemoryEfficientDataGenerator(Sequence):
    def __init__(self, indices, img_loader, meta_data, text1, text2, labels, batch_size=4):
        self.indices = indices
        self.img_loader = img_loader
        self.meta_data = tf.cast(meta_data, tf.float32)  # Ensure float32
        self.text1 = tf.cast(text1, tf.int32)  # Ensure int32
        self.text2 = tf.cast(text2, tf.int32)  # Ensure int32
        self.labels = tf.cast(labels, tf.float32)  # Ensure float32
        self.batch_size = batch_size
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Load images using the lazy loader
        x_img = np.array([self.img_loader[i] for i in batch_indices])
        x_img = tf.cast(x_img, tf.float32)  # Ensure float32
        x_img = tf.clip_by_value(x_img, 0.0, 1.0)  # Clip image values

        # Convert batch_indices to tensor for proper indexing
        batch_indices_tensor = tf.convert_to_tensor(batch_indices, dtype=tf.int32)

        # Get other features for the batch using gather_nd
        x_meta = tf.gather(self.meta_data, batch_indices_tensor)
        x_meta = tf.clip_by_value(x_meta, -1.0, 1.0)  # Clip metadata values

        x_txt1 = tf.gather(self.text1, batch_indices_tensor)
        x_txt2 = tf.gather(self.text2, batch_indices_tensor)
        y = tf.gather(self.labels, batch_indices_tensor)

        return (x_meta, x_txt1, x_txt2, x_img), y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

def create_memory_efficient_dataset(generator):
    return tf.data.Dataset.from_generator(
        lambda: generator,
        output_signature=(
            (
                tf.TensorSpec(shape=(None, data_meta.shape[1]), dtype=tf.float16),
                tf.TensorSpec(shape=(None, maxlen), dtype=tf.int32),
                tf.TensorSpec(shape=(None, maxlen), dtype=tf.int32),
                tf.TensorSpec(shape=(None, *im_size, 3), dtype=tf.float16)
            ),
            tf.TensorSpec(shape=(None, n_class), dtype=tf.float32)
        )
    ).prefetch(1)  # Reduced prefetch buffer

# ===========================
# Data Preparation
# ===========================

# Load and prepare data
print("Loading data...")
data_main = pd.read_csv('data_clean.csv')
data_main2 = pd.read_csv('clean_housing5.csv')
data2 = pd.read_csv('sentiment.csv')

# Preprocess data
print("Preprocessing data...")
data1 = data_main[['AR18', 'AR19']]
data1 = pd.get_dummies(data1, columns=data1.columns, drop_first=True)

# Merge datasets
merged_data = pd.concat([data2], axis=1)

# Create labels
data_main['lbl'] = data_main['AR166'].apply(lambda x: 0 if x == 0 else 1)

# Select balanced classes
idx_class_0 = np.where(data_main['lbl'].values == 0)[0]
idx_class_1 = np.where(data_main['lbl'].values == 1)[0][:14000]  # Limit class 1 samples

# Combine indices
selected_indices = np.concatenate([idx_class_0, idx_class_1])
data_main = data_main.iloc[selected_indices]
merged_data = merged_data.iloc[selected_indices]

# Prepare metadata
data_meta = merged_data
df_numeric = data_meta.values
scaler = MinMaxScaler()
scaler.fit(df_numeric)
df_numeric_normal = scaler.transform(df_numeric)

# Prepare labels
lbl_binary = tf.keras.utils.to_categorical(data_main["lbl"].values).astype(int)
n_class = lbl_binary.shape[1]

# Split data into train, validation, and test sets
train_Percentage = 0.8
all_indices = np.arange(len(lbl_binary))

# First split: train+val vs test
train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=1-train_Percentage,
    random_state=42
)

# Second split: train vs val
train_idx, valid_idx = train_test_split(
    train_val_idx,
    test_size=0.2,  # 20% of train+val for validation
    random_state=42
)

# Initialize lazy image loader
print("Initializing image loader...")
img_loader = LazyImageLoader(
    image_dir="data_image/first_class",
    image_names=data_main["image name"].values,
    target_size=im_size
)

# Prepare BERT inputs
print("Preparing BERT inputs...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def prepare_bert_input(text_data):
    return tokenizer(
        text_data.astype(str).tolist(),
        max_length=maxlen,
        truncation=True,
        padding="max_length",
        return_tensors="tf"
    )

train_input_bert = prepare_bert_input(data_main.iloc[train_idx]["text_clean2"])
valid_input_bert = prepare_bert_input(data_main.iloc[valid_idx]["text_clean2"])
test_input_bert = prepare_bert_input(data_main.iloc[test_idx]["text_clean2"])

train_input_bert2 = prepare_bert_input(data_main2.iloc[train_idx]["text_clean2"])
valid_input_bert2 = prepare_bert_input(data_main2.iloc[valid_idx]["text_clean2"])
test_input_bert2 = prepare_bert_input(data_main2.iloc[test_idx]["text_clean2"])

# Initialize generators with lazy loading
print("Initializing data generators...")
train_gen = MemoryEfficientDataGenerator(
    train_idx,
    img_loader=img_loader,
    meta_data=df_numeric_normal,
    text1=train_input_bert['input_ids'],
    text2=train_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=batch_size
)

valid_gen = MemoryEfficientDataGenerator(
    valid_idx,
    img_loader=img_loader,
    meta_data=df_numeric_normal,
    text1=valid_input_bert['input_ids'],
    text2=valid_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=batch_size
)

test_gen = MemoryEfficientDataGenerator(
    test_idx,
    img_loader=img_loader,
    meta_data=df_numeric_normal,
    text1=test_input_bert['input_ids'],
    text2=test_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=batch_size
)

# Create memory-efficient datasets
print("Creating datasets...")
train_dataset = create_memory_efficient_dataset(train_gen)
valid_dataset = create_memory_efficient_dataset(valid_gen)
test_dataset = create_memory_efficient_dataset(test_gen)

# ===========================
# Model Definition
# ===========================

print("Building model architecture...")

# Custom BERT Layer
class BertLayer(tf.keras.layers.Layer):
    def __init__(self, model_name="bert-base-uncased", **kwargs):
        super().__init__(**kwargs)
        self.bert = TFBertModel.from_pretrained(model_name)

    def call(self, inputs):
        return self.bert(inputs).last_hidden_state[:, 0, :]  # CLS token

# Shared Components
def create_qformer():
    inputs = Input(shape=(None, 24))  # Reduced from 96
    x = MultiHeadAttention(num_heads=1, key_dim=8)(inputs, inputs)  # Reduced parameters
    x = LayerNormalization()(x + inputs)
    return Model(inputs, x)

# Browse Phase (M_B)
def build_browse_model():
    input_meta = Input(shape=(data_meta.shape[1], 1), name="browse_meta")
    input_text1 = Input(shape=(maxlen,), dtype=tf.int32, name="browse_text1")
    input_image = Input(shape=im_size_final, name="browse_image")

    # Metadata processing with minimal parameters
    m = Conv1D(
        filters=8,  # Reduced from 32
        kernel_size=2,
        activation="relu"
    )(input_meta)
    m = GlobalAvgPool1D()(m)

    # Text processing with minimal parameters
    text_emb = BertLayer()(input_text1)
    text_proj = Dense(8)(text_emb)  # Reduced from 32

    # Image processing with minimal parameters
    vgg = VGG19(include_top=False, weights='imagenet')(input_image)
    img_feat = Dense(8)(GlobalAvgPool2D()(vgg))  # Reduced from 32

    # Combine features
    combined = Concatenate()([m, text_proj, img_feat])
    combined = Reshape((1, 24))(combined)  # Reduced from 96

    # Generate context
    context = create_qformer()(combined)
    context = GlobalAvgPool1D()(context)

    return Model(
        inputs=[input_meta, input_text1, input_image],
        outputs=context,
        name="M_B"
    )

# Concentrate Phase (M_C)
def build_concentrate_model(browse_model):
    input_meta = Input(shape=(data_meta.shape[1], 1), name="conc_meta")
    input_text1 = Input(shape=(maxlen,), dtype=tf.int32, name="conc_text1")
    input_text2 = Input(shape=(maxlen,), dtype=tf.int32, name="conc_text2")
    input_image = Input(shape=im_size_final, name="conc_image")

    # Get context vector
    context = browse_model([input_meta, input_text1, input_image])
    context = Reshape((1, 24))(context)  # Reduced from 96

    # Metadata processing with minimal parameters
    m = Conv1D(8, 2, activation="relu")(input_meta)  # Reduced from 32
    m = MultiHeadAttention(num_heads=1, key_dim=8)(  # Reduced heads and dim
        query=Add()([m, Dense(8)(context)]),
        value=m,
        key=m
    )
    m = GlobalAvgPool1D()(m)

    # Text processing with minimal parameters
    text_emb = BertLayer()(input_text2)
    context_proj = Dense(8)(context)  # Reduced from 32
    context_squeezed = Lambda(lambda x: K.squeeze(x, axis=1))(context_proj)
    text_feat = Dense(8)(text_emb) + context_squeezed  # Reduced from 32

    # Image processing with minimal parameters
    vgg = VGG19(include_top=False, weights='imagenet')(input_image)
    img_feat = Dense(8)(GlobalAvgPool2D()(vgg))  # Reduced from 32
    img_feat = Reshape((1, 8))(img_feat)
    img_feat = MultiHeadAttention(num_heads=1, key_dim=8)(  # Reduced heads and dim
        query=Add()([img_feat, Dense(8)(context)]),
        value=img_feat,
        key=img_feat
    )
    img_feat = GlobalAvgPool1D()(img_feat)

    # Final fusion
    fused = Concatenate()([m, text_feat, img_feat])
    output = Dense(n_class, activation="softmax")(fused)

    return Model(
        inputs=[input_meta, input_text1, input_text2, input_image],
        outputs=output,
        name="M_C"
    )

# Initialize and compile with memory optimizations
browse_model = build_browse_model()
concentrate_model = build_concentrate_model(browse_model)
browse_model.trainable = False

# Compile with memory-efficient optimizer and add gradient clipping
concentrate_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5,  # Reduced learning rate
        clipnorm=0.5,  # More conservative gradient clipping
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Memory cleanup callback
class MemoryCleanupCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        tf.keras.backend.clear_session()

# Add a custom callback to monitor and handle NaN values
class NanMonitorCallback(tf.keras.callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        if logs is not None:
            if np.isnan(logs.get('loss', 0)):
                print(f"NaN detected in loss at batch {batch}")
                # Instead of stopping, try to recover
                self.model.optimizer.learning_rate.assign(self.model.optimizer.learning_rate * 0.5)
                print(f"Reduced learning rate to {self.model.optimizer.learning_rate.numpy()}")

# Train the model with memory optimizations and NaN monitoring
history = concentrate_model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=2,
    steps_per_epoch=len(train_gen),
    validation_steps=len(valid_gen),
    verbose=1,
    callbacks=[
        MemoryCleanupCallback(),
        NanMonitorCallback(),  # Add NaN monitoring
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=2,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=1,
            min_lr=1e-7  # Reduced minimum learning rate
        )
    ]
)

# ===========================
# Standard Libraries
# ===========================

import os  # Provides functions for interacting with the operating system
import re  # Regular expressions for string manipulation
from pathlib import Path  # Object-oriented file system paths
import shutil  # File operations such as copying and moving files

# TensorFlow Callbacks and Mixed Precision
from tensorflow.keras.callbacks import LambdaCallback  # Allows dynamic behavior during training
from tensorflow.keras import mixed_precision  # Enables mixed precision training
# mixed_precision.set_global_policy('mixed_float16')  # Uncomment to enable mixed precision (faster computation on supported hardware)

# ===========================
# Data Manipulation and Analysis
# ===========================

import pandas as pd  # Library for data analysis and manipulation
import numpy as np  # Numerical computations and array handling

# ===========================
# Data Visualization
# ===========================

import matplotlib.pyplot as plt  # Visualization library
import seaborn as sns  # Enhances Matplotlib with statistical plotting
from wordcloud import WordCloud, STOPWORDS  # For generating word clouds
from mlxtend.plotting import plot_confusion_matrix  # Function for plotting confusion matrices

# ===========================
# Deep Learning and TensorFlow
# ===========================

import tensorflow as tf  # Core deep learning library
from tensorflow.keras.layers import (  # Keras layers for deep learning models
    Concatenate, MultiHeadAttention, LayerNormalization,
    Dense, Input, Reshape, GlobalAvgPool2D, GlobalAvgPool1D,
    Conv1D, Add, Lambda, Layer
)
import keras.ops as K  # Keras operations module for mathematical functions

# Pretrained CNN Models
from tensorflow.keras.applications import VGG19  # Pretrained VGG19 model for image processing

# Keras Functional API for Model Creation
from tensorflow.keras.models import Model  # Keras model-building API

# Transformers (BERT)
from transformers import TFBertModel  # Pretrained BERT model for NLP tasks

# ===========================
# Machine Learning and Preprocessing
# ===========================

from sklearn import preprocessing  # Data preprocessing utilities
from sklearn.preprocessing import MinMaxScaler  # Feature scaling method
from sklearn.model_selection import train_test_split  # Splitting datasets into train/validation/test
from sklearn.utils import class_weight  # Handling imbalanced datasets

# Metrics for Model Evaluation
from sklearn.metrics import (
    classification_report,  # Generates detailed classification performance reports
    accuracy_score,  # Computes accuracy of predictions
    confusion_matrix,  # Generates a confusion matrix
    roc_auc_score,  # Computes ROC AUC score for classification
    average_precision_score  # Computes average precision for precision-recall curves
)

# ===========================
# Deep Learning - Keras & TensorFlow
# ===========================

import tensorflow as tf  # TensorFlow deep learning framework
from tensorflow import keras  # High-level deep learning API

# Keras Functional API for Building Models
from tensorflow.keras.models import Model  # Model-building API
from tensorflow.keras.layers import (  # Import additional layers
    MultiHeadAttention, LayerNormalization,
    Dense, Input, Reshape, Flatten
)
from tensorflow.keras.utils import to_categorical  # Converts labels to one-hot encoding

# ===========================
# Preprocessing for Images and Text
# ===========================

# Image Processing
from tensorflow.keras.preprocessing.image import ImageDataGenerator as keras_image  # Data augmentation and preprocessing for images

# Tokenization for Text Data
from tensorflow.keras.preprocessing.text import Tokenizer  # Tokenizer for converting text to sequences
# from keras_preprocessing.sequence import pad_sequences  # Pad sequences to the same length

# Image Augmentation
from keras_preprocessing.image import ImageDataGenerator  # Data augmentation for training images

# ===========================
# NLP - Transformers (BERT)
# ===========================

from transformers import BertTokenizer, TFBertModel  # Pretrained BERT tokenizer and model for NLP tasks

# Print a confirmation message
print("All required libraries successfully imported.")

"""#Importing Libraries

#Data Preprocessing Part
"""

# Load datasets
data_main = pd.read_csv('data_clean.csv')
data_main2 = pd.read_csv('clean_housing5.csv')
data2 = pd.read_csv('sentiment.csv')

# Preprocess data
data1 = data_main[['AR18', 'AR19']]
data1 = pd.get_dummies(data1, columns=data1.columns, drop_first=True)

# Merge datasets
merged_data = pd.concat([data2], axis=1)

# Create labels
data_main['lbl'] = data_main['AR166'].apply(lambda x: 0 if x == 0 else 1)

# Analyze class distribution
class_count = data_main["lbl"].value_counts().values
lst_class = data_main["lbl"].value_counts().index.to_list()
lst_class = ["class " + str(item) for item in lst_class]

# Plot class distribution
plt.figure(figsize=(6, 6))
palette_color = sns.color_palette('muted')
plt.pie(
    class_count,
    labels=lst_class,
    colors=palette_color,
    autopct='%.0f%%',
    textprops={'fontsize': 14}
)
plt.title('Number of each class')
plt.show()

# class 0 select
idx_class=np.argwhere(data_main['lbl'].values==0)
idx_class_0=[item[0] for item in idx_class]
idx_class_0=idx_class_0

# class 1 select
idx_class=np.argwhere(data_main['lbl'].values==1)
idx_class_1=[item[0] for item in idx_class]
idx_class_1=idx_class_1[0:14000]

# all select
idx_class_0.extend(idx_class_1)
data_main=data_main.iloc[idx_class_0]

merged_data=merged_data.iloc[idx_class_0]

# class 0 select
idx_class=np.argwhere(data_main2['lbl'].values==0)
idx_class_0=[item[0] for item in idx_class]
idx_class_0=idx_class_0

# class 1 select
idx_class=np.argwhere(data_main2['lbl'].values==1)
idx_class_1=[item[0] for item in idx_class]
idx_class_1=idx_class_1[0:14000]

# all select
idx_class_0.extend(idx_class_1)
data_main2=data_main2.iloc[idx_class_0]

data_meta=merged_data

df_numeric=data_meta.values

scaler = MinMaxScaler()
scaler.fit(df_numeric)
df_numeric_normal = scaler.transform(df_numeric)

data_pre=data_main.copy()

im_size = (224,224)
input_size=(im_size[0],im_size[1],3)
batch_value=512

"""##Moving images to data_image/first_class folder"""

folder_name = Path("data_image/first_class")
folder_name.mkdir(parents=True, exist_ok=True)  # Creates folder if it doesn't exist
print(f"Folder '{folder_name}' created successfully!")

# Define source directory (current directory) and destination folder
source_dir = Path(".")  # Current directory
destination_folder = Path("data_image/first_class")

# Ensure the destination folder exists
destination_folder.mkdir(parents=True, exist_ok=True)

# Move all .png files
for png_file in source_dir.glob("*.png"):
    shutil.move(str(png_file), str(destination_folder / png_file.name))
    print(f"Moved: {png_file} -> {destination_folder / png_file.name}")

print("All .png images have been moved successfully!")

# Define the image directory path
image_path = 'data_image/'

# Initialize ImageDataGenerator for preprocessing images (rescaling pixel values)
image_generator = ImageDataGenerator(rescale=1./255)

# Load images from the directory, resizing them and setting batch size
image_set = image_generator.flow_from_directory(image_path,
                                                target_size=im_size,  # 'im_size' should be defined elsewhere
                                                batch_size=70295)  # Batch size should match dataset size

# Extract image data (X) and labels (y) from the dataset
X_image, y_image = image_set.__next__()

# Print the shape of the extracted image data
print(X_image.shape)

# Extract numerical indices from image filenames
idx_img = np.array([int(re.findall(r'\b\d+\b', item)[0]) for item in image_set.filenames])
idx_img.T  # Transpose the array (though not necessary here)

# Rename column 'AR129' to 'image name' in data_pre DataFrame
data_pre = data_pre.rename(columns={'AR129': 'image name'})

# Replace all occurrences of 0 in the 'image name' column with 10
data_pre["image name"][data_pre["image name"] == 0] = 10

# Initialize an empty list to store processed images
img_list = []

# Loop through image names in the dataset and extract corresponding images
for item in data_pre["image name"]:
    idx_im_find = np.argwhere(idx_img == item)[0][0]  # Find the index of the image
    img = X_image[idx_im_find]  # Get the image data
    if isinstance(img, np.ndarray):
        if img.shape != (*im_size, 3):
            img = cv2.resize(img, im_size)
        if len(img.shape) == 2:  # If grayscale, convert to RGB
            img = np.stack([img] * 3, axis=-1)
        img_list.append(img.astype(np.float32))  # Store as float32
    else:
        # If image is invalid, create a blank image
        img_list.append(np.zeros((*im_size, 3), dtype=np.float32))

# Convert to numpy array for faster indexing
img_list = np.array(img_list)

print(f"Loaded {len(img_list)} images with shape {img_list[0].shape}")

# Extract text tokens from the dataset for word cloud visualization
data_token = data_pre["text_clean2"].values
data_token2 = data_main2["text_clean2"].values  # 'data_main2' should be defined earlier

# Generate and display a word cloud from the text data
plt.figure(figsize=(8, 8))
text = data_token
wordcloud = WordCloud().generate(str(text))

# Copy the tokenized text data
data_token_re = data_token.copy()

import numpy as np
from keras_preprocessing.sequence import pad_sequences as original_pad_sequences

def pad_sequences(sequences, maxlen=None, dtype='int32',
                  padding='pre', truncating='pre', value=0.):
    """Pads sequences to the same length.

    This function transforms a list (of length `num_samples`)
    of sequences (lists of integers)
    into a 2D Numpy array of shape `(num_samples, num_timesteps)`.
    `num_timesteps` is either the `maxlen` argument if provided,
    or the length of the longest sequence in the list.

    Sequences that are shorter than `num_timesteps`
    are padded with `value` at either the beginning or the end.
    Sequences longer than `num_timesteps` are truncated
    so that they fit the desired length.
    The position where padding or truncation happens is determined by
    the arguments `padding` and `truncating`, respectively.
    Pre-padding means that zeros are added to the beginning of the sequence,
    while post-padding means that zeros are added to the end.
    Pre-truncating means that the end of the sequence is removed,
    while post-truncating means that the beginning of the sequence is removed.

    # Arguments
        sequences: List of lists of int or float.
        maxlen: None or int. Maximum sequence length.
            If None, will use the maximum length in the sequences.
        dtype: Type of the output sequences.
            To pad sequences with variable length strings, you can use `object`.
        padding: 'pre' or 'post', pad either before or after each sequence.
        truncating: 'pre' or 'post', remove values from sequences larger than
            `maxlen` either before or after.
        value: Float or String, padding value.

    # Returns
        x: Numpy array with shape `(len(sequences), maxlen)`

    # Raises
        ValueError: In case of invalid values for `truncating` or `padding`,
            or in case of invalid shape for a `sequences` entry.
    """
    if maxlen is None:
        maxlen = np.max(lengths)

    # Check for dtype compatibility with 'value'
    is_dtype_str = np.issubdtype(dtype, np.str_) or np.issubdtype(dtype, np.str_)  # Modified line
    if isinstance(value, str) and dtype != object and not is_dtype_str:
        raise ValueError("`dtype` {} is not compatible with `value`'s type: {}\n"
                         "You should set `dtype=object` for variable length "
                         "strings.".format(dtype, type(value)))

    x = np.asarray([[value for _ in range(maxlen)] for _ in range(len(sequences))],
                    dtype=dtype)
    for idx, s in enumerate(sequences):
        if not len(s):
            continue  # empty list/array was found
        if truncating == 'pre':
            trunc = s[-maxlen:]
        elif truncating == 'post':
            trunc = s[:maxlen]
        else:
            raise ValueError('Truncating type "%s" '
                             'not understood' % truncating)

        if padding == 'pre':
            x[idx, -len(trunc):] = np.asarray(trunc, dtype=dtype)
        elif padding == 'post':
            x[idx, :len(trunc)] = np.asarray(trunc, dtype=dtype)
        else:
            raise ValueError('Padding type "%s" not understood' % padding)
    return x

# Define the maximum number of words to keep in the tokenizer's vocabulary
MAX_NB_WORDS = 99826

# Initialize the tokenizer with the specified vocabulary size
tokenizer = Tokenizer(num_words=MAX_NB_WORDS)

# Fit the tokenizer on the preprocessed text data
tokenizer.fit_on_texts(data_token_re)

# Convert the text data into sequences of numerical values
data_seq = tokenizer.texts_to_sequences(data_token_re)

# Calculate the vocabulary size (adding 1 because index 0 is reserved)
vocab_size = len(tokenizer.word_index) + 1

# Print the last preprocessed text entry and its corresponding sequence
print(data_token_re[-1])
print(data_seq[-1])

# Calculate the length of each sequence (number of words per entry)
frq_words = [len(item) for item in data_seq]

# Define the maximum sequence length (for padding)
maxlen = 500

# Pad sequences to ensure uniform length, truncating or padding with 0 as needed
data_seq_pad = pad_sequences(data_seq, padding='post', maxlen=maxlen)

# Print the first 20 elements of the 10th padded sequence
print(data_seq_pad[10, 0:20])

# Convert labels to a binary format using one-hot encoding
lbl_binary = to_categorical(data_pre["lbl"].values).astype(int)

# Define the percentage of data to use for training
train_Percentage = 0.90

# path: data_generators.py
import numpy as np
import cv2
from tensorflow.keras.utils import Sequence

class MultiInputDataGenerator(Sequence):
    def __init__(self, indices, img_list, meta_data, text1, text2, labels, batch_size=32, im_size=(224, 224), shuffle=True):
        self.indices = np.array(indices)
        self.img_list = img_list  # Keep as reference only
        self.meta_data = meta_data
        self.text1 = text1
        self.text2 = text2
        self.labels = labels
        self.batch_size = batch_size
        self.im_size = im_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Process images in batch with memory optimization
        x_img = np.zeros((len(batch_indices), *self.im_size, 3), dtype=np.float32)
        for i, idx in enumerate(batch_indices):
            img = self.img_list[idx]
            if isinstance(img, np.ndarray):
                if img.shape != (*self.im_size, 3):
                    img = cv2.resize(img, self.im_size)
                if len(img.shape) == 2:
                    img = np.stack([img] * 3, axis=-1)
                x_img[i] = img.astype(np.float32) / 255.0

        # Get other features for the batch
        x_meta = self.meta_data[batch_indices]
        x_txt1 = self.text1[batch_indices]
        x_txt2 = self.text2[batch_indices]
        y = self.labels[batch_indices]

        return (x_meta, x_txt1, x_txt2, x_img), y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def get_output_signature(self):
        """Returns the output signature for the dataset."""
        return (
            (
                tf.TensorSpec(shape=(None, self.meta_data.shape[1]), dtype=tf.float32),
                tf.TensorSpec(shape=(None, self.text1.shape[1]), dtype=tf.int32),
                tf.TensorSpec(shape=(None, self.text2.shape[1]), dtype=tf.int32),
                tf.TensorSpec(shape=(None, *self.im_size, 3), dtype=tf.float32)
            ),
            tf.TensorSpec(shape=(None, self.labels.shape[1]), dtype=tf.float32)
        )

from sklearn.model_selection import train_test_split
import numpy as np

# Define indices only once
all_indices = np.arange(len(lbl_binary))

# Split indices into train and test
train_val_idx, test_idx, lbl_trainval, lbl_test = train_test_split(
    all_indices,
    lbl_binary,
    test_size=1 - train_Percentage,
    random_state=0
)

# Split train indices into train and validation
train_idx, valid_idx, lbl_train, lbl_valid = train_test_split(
    train_val_idx,
    lbl_trainval,
    test_size=1 - train_Percentage,
    random_state=0
)

# Now apply indices to fetch all input components lazily later
x_train_idx = train_idx
x_valid_idx = valid_idx
x_test_idx = test_idx

# Print data percentage using just the indices
print('x_train count: ', len(x_train_idx), np.round(len(x_train_idx) / len(lbl_binary), 2), '%')
print('x_valid count: ', len(x_valid_idx), np.round(len(x_valid_idx) / len(lbl_binary), 2), '%')
print('x_test count:  ', len(x_test_idx), np.round(len(x_test_idx) / len(lbl_binary), 2), '%')

# You can pass these indices to generators:
# e.g., MultiInputDataGenerator(x_train_idx, img_paths, x_meta, input_bert, input_bert2, lbl_binary, ...)

# Use same indices to extract all other input types when needed
x_train_TEXT  = data_token[x_train_idx]
x_valid_TEXT  = data_token[x_valid_idx]
x_test_TEXT   = data_token[x_test_idx]

x_train_TEXT2 = data_token2[x_train_idx]
x_valid_TEXT2 = data_token2[x_valid_idx]
x_test_TEXT2  = data_token2[x_test_idx]

x_train_meta  = df_numeric_normal[x_train_idx]
x_valid_meta  = df_numeric_normal[x_valid_idx]
x_test_meta   = df_numeric_normal[x_test_idx]

# Keep image data path or access it on demand by index from img_list
x_train_img_paths = [img_list[i] for i in x_train_idx]
x_valid_img_paths = [img_list[i] for i in x_valid_idx]
x_test_img_paths  = [img_list[i] for i in x_test_idx]

# Define evaluation metrics for the model
METRICS = [
    keras.metrics.TruePositives(name='tp'),  # Number of true positives
    keras.metrics.FalsePositives(name='fp'),  # Number of false positives
    keras.metrics.TrueNegatives(name='tn'),  # Number of true negatives
    keras.metrics.FalseNegatives(name='fn'),  # Number of false negatives
    keras.metrics.CategoricalAccuracy(name='accuracy'),  # Overall classification accuracy
    keras.metrics.Precision(name='precision'),  # Precision metric
    keras.metrics.Recall(name='recall'),  # Recall metric
    keras.metrics.AUC(name='auc'),  # Area under the ROC curve
    keras.metrics.AUC(name='prc', curve='PR')  # Area under the precision-recall curve
]

# Compute class weights for handling imbalanced data
class_weights_val = class_weight.compute_class_weight(
    class_weight='balanced',  # Balance class weights based on sample frequency
    classes=np.unique(np.argmax(lbl_valid, axis=1)),  # Unique class labels
    y=np.argmax(lbl_valid, axis=1)  # Convert one-hot labels to class indices
)

# Convert computed class weights into a dictionary format required by Keras
class_weight = dict()
for idx, val in enumerate(class_weights_val):
    class_weight[idx] = val  # Assign weight value to the corresponding class index

# Define the final image size with 3 color channels (RGB)
im_size_final = (im_size[0], im_size[1], 3)

# Print computed class weights and final image size for verification
print("Class Weights:", class_weight)
print("Final Image Size:", im_size_final)

# Define key parameters
embedding_dim = 50  # Dimension of embedding layer (not used in BERT but might be useful for other models)
n_class = 2  # Number of output classes
maxlen_bert = maxlen  # Maximum sequence length for BERT input

# Define strategy for distributed training (useful for multi-GPU setups)
strategy = tf.distribute.MirroredStrategy()

# Create model and tokenizer within the distributed strategy scope
with strategy.scope():
    # Load pre-trained BERT model from Hugging Face
    bert_layer = TFBertModel.from_pretrained("bert-base-uncased")

    # Initialize tokenizer for BERT
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def bert_encode(texts, tokenizer, max_len=512):
    """
    Tokenizes and encodes input texts for BERT.

    Parameters:
        texts (list or array): Input text samples.
        tokenizer (BertTokenizer): Pre-trained BERT tokenizer.
        max_len (int): Maximum length for tokenized sequences.

    Returns:
        dict: Encoded text with input IDs, attention masks, and token type IDs.
    """
    encoded = tokenizer(
        texts,
        max_length=max_len,  # Set maximum sequence length
        truncation=True,  # Truncate longer texts
        padding="max_length",  # Pad shorter texts to max_len
        return_tensors="tf",  # Return TensorFlow tensors
    )
    return {
        "input_ids": encoded["input_ids"],  # Token IDs
        "attention_mask": encoded["attention_mask"],  # Attention masks (1 for real tokens, 0 for padding)
        "token_type_ids": encoded["token_type_ids"],  # Segment IDs for sentence pairs (not always needed)
    }

# Print confirmation that BERT model and tokenizer are loaded
print("BERT model and tokenizer successfully loaded.")

# Use the same distributed training strategy
with strategy.scope():
    # Encode the training text data using the BERT tokenizer
    train_input_bert = bert_encode(
        x_train_TEXT.astype(str).tolist(),  # Convert input text to a list of strings
        tokenizer,  # Use the pre-trained tokenizer
        max_len=maxlen_bert  # Define the maximum sequence length
    )

    # Encode the validation text data
    valid_input_bert = bert_encode(
        x_valid_TEXT.astype(str).tolist(),
        tokenizer,
        max_len=maxlen_bert
    )

    # Encode the test text data
    test_input_bert = bert_encode(
        x_test_TEXT.astype(str).tolist(),
        tokenizer,
        max_len=maxlen_bert
    )

    # Encode the second set of training text data
    train_input_bert2 = bert_encode(
        x_train_TEXT2.astype(str).tolist(),
        tokenizer,
        max_len=maxlen_bert
    )

    # Encode the second set of validation text data
    valid_input_bert2 = bert_encode(
        x_valid_TEXT2.astype(str).tolist(),
        tokenizer,
        max_len=maxlen_bert
    )

    # Encode the second set of test text data
    test_input_bert2 = bert_encode(
        x_test_TEXT2.astype(str).tolist(),
        tokenizer,
        max_len=maxlen_bert
    )

# Print confirmation message
print("BERT input encoding completed for training, validation, and test datasets.")

im_size_final=im_size[0],im_size[1],3

input_meta = Input(shape=(data_meta.shape[1], 1), name="browse_meta")

# Custom BERT Layer (ignore PyTorch weight warnings)
class BertLayer(Layer):
    def __init__(self, model_name="bert-base-uncased", **kwargs):
        super().__init__(**kwargs)
        self.bert = TFBertModel.from_pretrained(model_name)

    def call(self, inputs):
        return self.bert(inputs).last_hidden_state[:, 0, :]  # CLS token

# ------------------- Shared Components ------------------- #
def create_qformer():
    inputs = Input(shape=(None, 96))  # Input shape: (batch, seq_len, features)
    x = MultiHeadAttention(num_heads=2, key_dim=32)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    return Model(inputs, x)

# ------------------- Browse Phase (M_B) ------------------- #
def build_browse_model():
    # Ensure data_meta has valid shape
    input_meta = Input(shape=(data_meta.shape[1], 1), name="browse_meta")
    input_text1 = Input(shape=(maxlen,), dtype=tf.int32, name="browse_text1")
    input_image = Input(shape=im_size_final, name="browse_image")

    # Metadata processing with minimal parameters
    m = Conv1D(
        filters=8,  # Reduced from 32
        kernel_size=2,
        activation="relu"
    )(input_meta)
    m = GlobalAvgPool1D()(m)

    # Text processing with minimal parameters
    text_emb = BertLayer()(input_text1)
    text_proj = Dense(8)(text_emb)  # Reduced from 32

    # Image processing with minimal parameters
    vgg = VGG19(include_top=False, weights='imagenet')(input_image)
    img_feat = Dense(8)(GlobalAvgPool2D()(vgg))  # Reduced from 32

    # Combine features
    combined = Concatenate()([m, text_proj, img_feat])
    combined = Reshape((1, 24))(combined)  # Reduced from 96

    # Generate context
    context = create_qformer()(combined)
    context = GlobalAvgPool1D()(context)

    return Model(
        inputs=[input_meta, input_text1, input_image],
        outputs=context,
        name="M_B"
    )

# ------------------- Concentrate Phase (M_C) ------------------- #
def build_concentrate_model(browse_model):
    input_meta = Input(shape=(data_meta.shape[1], 1), name="conc_meta")
    input_text1 = Input(shape=(maxlen,), dtype=tf.int32, name="conc_text1")
    input_text2 = Input(shape=(maxlen,), dtype=tf.int32, name="conc_text2")
    input_image = Input(shape=im_size_final, name="conc_image")

    # Get context vector
    context = browse_model([input_meta, input_text1, input_image])
    context = Reshape((1, 24))(context)  # Reduced from 96

    # Metadata processing with minimal parameters
    m = Conv1D(8, 2, activation="relu")(input_meta)  # Reduced from 32
    m = MultiHeadAttention(num_heads=1, key_dim=8)(  # Reduced heads and dim
        query=Add()([m, Dense(8)(context)]),
        value=m,
        key=m
    )
    m = GlobalAvgPool1D()(m)

    # Text processing with minimal parameters
    text_emb = BertLayer()(input_text2)
    context_proj = Dense(8)(context)  # Reduced from 32
    context_squeezed = Lambda(lambda x: K.squeeze(x, axis=1))(context_proj)
    text_feat = Dense(8)(text_emb) + context_squeezed  # Reduced from 32

    # Image processing with minimal parameters
    vgg = VGG19(include_top=False, weights='imagenet')(input_image)
    img_feat = Dense(8)(GlobalAvgPool2D()(vgg))  # Reduced from 32
    img_feat = Reshape((1, 8))(img_feat)
    img_feat = MultiHeadAttention(num_heads=1, key_dim=8)(  # Reduced heads and dim
        query=Add()([img_feat, Dense(8)(context)]),
        value=img_feat,
        key=img_feat
    )
    img_feat = GlobalAvgPool1D()(img_feat)

    # Final fusion
    fused = Concatenate()([m, text_feat, img_feat])
    output = Dense(n_class, activation="softmax")(fused)

    return Model(
        inputs=[input_meta, input_text1, input_text2, input_image],
        outputs=output,
        name="M_C"
    )

# Initialize and compile with memory optimizations
browse_model = build_browse_model()
concentrate_model = build_concentrate_model(browse_model)
browse_model.trainable = False

# Compile with memory-efficient optimizer and add gradient clipping
concentrate_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5,  # Reduced learning rate
        clipnorm=0.5,  # More conservative gradient clipping
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

train_gen = MultiInputDataGenerator(
    x_train_idx,
    img_list=img_list,
    meta_data=df_numeric_normal,
    text1=train_input_bert['input_ids'],
    text2=train_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=16,
    im_size=im_size_final
)

valid_gen = MultiInputDataGenerator(
    x_valid_idx,
    img_list=img_list,
    meta_data=df_numeric_normal,
    text1=valid_input_bert['input_ids'],
    text2=valid_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=16,
    im_size=im_size_final,
    shuffle=False
)

test_gen = MultiInputDataGenerator(
    x_test_idx,
    img_list=img_list,
    meta_data=df_numeric_normal,
    text1=test_input_bert['input_ids'],
    text2=test_input_bert2['input_ids'],
    labels=lbl_binary,
    batch_size=16,
    im_size=im_size_final,
    shuffle=False
)

# Create datasets with proper output signatures and memory optimizations
def create_dataset(generator):
    return tf.data.Dataset.from_generator(
        lambda: generator,
        output_signature=generator.get_output_signature()
    ).prefetch(tf.data.AUTOTUNE) \
     .cache() \
     .shuffle(buffer_size=1000) \
     .batch(8)  # Reduced batch size

train_dataset = create_dataset(train_gen)
valid_dataset = create_dataset(valid_gen)
test_dataset = create_dataset(test_gen)

# Add memory cleanup between epochs
class MemoryCleanupCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        tf.keras.backend.clear_session()

# Train the model with memory optimizations and NaN monitoring
history = concentrate_model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=2,
    steps_per_epoch=len(train_gen),
    validation_steps=len(valid_gen),
    verbose=1,
    callbacks=[
        MemoryCleanupCallback(),
        NanMonitorCallback(),  # Add NaN monitoring
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=2,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=1,
            min_lr=1e-7  # Reduced minimum learning rate
        )
    ]
)

1 Physical GPUs, 1 Logical GPUs
Loading data...
Preprocessing data...
Initializing image loader...
Preparing BERT inputs...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Initializing data generators...
Creating datasets...
Building model architecture...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Epoch 1/2
 666/2607 ━━━━━━━━━━━━━━━━━━━━ 2:19:25 4s/step - accuracy: 0.8058 - loss: 0.6926

In [ ]:

print("Let's predict")

# Predict using the test generator
pred_test = concentrate_model.predict(test_dataset)
print("Finished predict")

# Convert predictions and labels to integer format
lbl_pred = np.argmax(pred_test, axis=1).astype(int)
lbl_real = np.argmax(lbl_test, axis=1).astype(int)

# Save predictions and labels
import pickle
with open('lbl_pred.p', 'wb') as f:
    pickle.dump(lbl_pred, f)

with open('lbl_real.p', 'wb') as f:
    pickle.dump(lbl_real, f)

with open('x_test_meta.p', 'wb') as f:
    pickle.dump(x_test_meta, f)

with open('x_test_TEXT.p', 'wb') as f:
    pickle.dump(x_test_TEXT, f)

# Plot training and validation loss
plt.figure(figsize=(8, 6))
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig('a4')

# Classification report and confusion matrix

classfi_report = classification_report(lbl_real, lbl_pred, output_dict=True)
accuracy = accuracy_score(lbl_real, lbl_pred)
precision = classfi_report['macro avg']['precision']
recall = classfi_report['macro avg']['recall']
f1_score = classfi_report['macro avg']['f1-score']
Con_matrix = confusion_matrix(lbl_real, lbl_pred)

# Plot confusion matrix
fig, ax = plot_confusion_matrix(
    conf_mat=Con_matrix,
    show_absolute=True,
    show_normed=True,
    colorbar=True,
    figsize=(n_class + 2, n_class + 2)
)
ax.set_title('Confusion Matrix of Browse-and-Concentrate Model')
plt.savefig('matrix1')

print("-" * 50)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1_score}")
print("-" * 50)

# Calculate AUC and pAUC
auc_score = roc_auc_score(lbl_real, lbl_pred)
pauc_score = average_precision_score(lbl_real, lbl_pred)

print("-" * 50)
print(f"AUC ROC: {auc_score}")
print(f"pAUC ROC: {pauc_score}")
print("-" * 50)

# Bootstrap function for confidence intervals
def BootstrapR2(pred_test, lbl_test, numboot=10000):
    lbl_pred = np.argmax(pred_test, axis=1).astype(int)
    lbl_real = np.argmax(lbl_test, axis=1).astype(int)
    classfi_report = classification_report(lbl_real, lbl_pred, output_dict=True)
    accuracy = accuracy_score(lbl_real, lbl_pred)
    precision = classfi_report['macro avg']['precision']
    recall = classfi_report['macro avg']['recall']
    f1_score = classfi_report['macro avg']['f1-score']
    roc = roc_auc_score(lbl_real, lbl_pred)
    pauc_score = average_precision_score(lbl_real, lbl_pred)

    # Bootstrap sampling
    n = len(lbl_test)
    roc1 = np.zeros((numboot, 1))
    roc2 = np.zeros((numboot, 1))
    f1_score1 = np.zeros((numboot, 1))
    accuracy1 = np.zeros((numboot, 1))
    precision1 = np.zeros((numboot, 1))
    recall1 = np.zeros((numboot, 1))
    pauc1 = np.zeros((numboot, 1))

    for i in range(numboot):
        random_index1 = np.random.randint(0, lbl_pred.shape[0], size=len(lbl_test))
        lbl_test1 = lbl_test[random_index1]
        pred_test1 = pred_test[random_index1]

        lbl_pred1 = np.argmax(pred_test1, axis=1).astype(int)
        lbl_real1 = np.argmax(lbl_test1, axis=1).astype(int)

        classfi_report = classification_report(lbl_real1, lbl_pred1, output_dict=True)

        accuracy1[i] = accuracy_score(lbl_real1, lbl_pred1)
        precision1[i] = classfi_report['macro avg']['precision']
        recall1[i] = classfi_report['macro avg']['recall']
        f1_score1[i] = classfi_report['macro avg']['f1-score']
        roc1[i] = roc_auc_score(lbl_real1, lbl_pred1)
        roc2[i] = roc_auc_score(lbl_test1, pred_test1)
        pauc1[i] = average_precision_score(lbl_real1, lbl_pred1)

    # Print results with confidence intervals
    print("-" * 50)
    print("AUC1")
    print(f"{round(roc, 3)} [{round(np.quantile(roc1 - roc, 0.025), 3)}, {round(np.quantile(roc1 - roc, 0.975), 3)}]")

    print("AUC2")
    print(f"{round(roc, 3)} [{round(np.quantile(roc2 - roc, 0.025), 3)}, {round(np.quantile(roc2 - roc, 0.975), 3)}]")

    print("F1 Score")
    print(f"{round(f1_score, 3)} [{round(np.quantile(f1_score1 - f1_score, 0.025), 3)}, {round(np.quantile(f1_score1 - f1_score, 0.975), 3)}]")

    print("Accuracy")
    print(f"{round(accuracy, 3)} [{round(np.quantile(accuracy1 - accuracy, 0.025), 3)}, {round(np.quantile(accuracy1 - accuracy, 0.975), 3)}]")

    print("Precision")
    print(f"{round(precision, 3)} [{round(np.quantile(precision1 - precision, 0.025), 3)}, {round(np.quantile(precision1 - precision, 0.975), 3)}]")

    print("Recall")
    print(f"{round(recall, 3)} [{round(np.quantile(recall1 - recall, 0.025), 3)}, {round(np.quantile(recall1 - recall, 0.975), 3)}]")

    print("pAUC")
    print(f"{round(pauc_score, 3)} [{round(np.quantile(pauc1 - pauc_score, 0.025), 3)}, {round(np.quantile(pauc1 - pauc_score, 0.975), 3)}]")

    return pauc1, roc1, f1_score1, accuracy1, precision1, recall1

# Run bootstrap analysis
pauc1, roc1, f1_score1, accuracy1, precision1, recall1 = BootstrapR2(pred_test, lbl_test, numboot=10000)